In [1]:
import math
from abc import ABC, abstractmethod
from typing import Self

import plotly.graph_objects as go
import torch
from jaxtyping import Float
from pydantic import ConfigDict
from torch import Tensor

from src.config.base import BaseConfig
from src.data.base import BaseDataConfig
from src.priors.base import BasePriorConfig

In [ ]:
class LagrangianConstraintConfig(BaseConfig):
    data_constraint_budget: float
    prior_constraint_budget: float

    dual_variable_lr: float

class LossConstraintConfig(BaseConfig):
    data_loss_weight: float
    prior_loss_weight: float

class ManifoldConstraintConfig(BaseConfig):
    huber_delta: float
    """Use huber loss to stabilize far-off-manifold conditions"""
    constraint_method: LagrangianConstraintConfig | LossConstraintConfig

In [4]:
class BaseKLWeightSchedule(BaseConfig, ABC):
    """
    Each class specifies a bulk-KL weighting.
    Bulk-kl weighting translates to score-matching weight along the noise spectrum
    """
    @abstractmethod
    def bulk_kl_weight(self, t: Float[Tensor, "batch"]) -> Float[Tensor, "batch"]:
        """Returns the de-Bruijn weighting of the score integral"""
        raise NotImplementedError

    def pullback_weight(self, t: Float[Tensor, "batch"]) -> Float[Tensor, "batch"]:
        """
        Pullbacks are multiplied with (1-t) interpolation weight.
        **Used to aggregate the transport field.**
        """
        return self.bulk_kl_weight(t) * (1.0 - t)


class UniformVelocityMatchingSchedule(BaseKLWeightSchedule):
    """
    Uniform velocity matching in flow-matching
    corresponds to a KL-integral with weight 1/(1-t)**2
    """
    def bulk_kl_weight(self, t: Float[Tensor, "batch"]) -> Float[Tensor, "batch"]:
        return (1 - t).pow(-2.)

In [ ]:
class TransportLossConfig(BaseConfig):
    """
    Specification for how the transport field is estimated and determine transport
    """
    kl_weight_schedule: BaseKLWeightSchedule
    """weight along the noise spectrum"""
    transport_constant: float
    """Target transport is computed as transport_constant * field"""
    num_time_samples: int
    """We approximate the actual transport-field integral across this many
    noise-spectrum time samples, with stratified uniform sampling within each bin"""
    antipodal_estimate: bool
    """Whether to use antipodal estimate to estimate the field"""
    decoder_transport_weight: float
    encoder_transport_weight: float
    """Weights for the transport losses"""

class ChartTransportLossConfig(BaseConfig):
    """
    Specifies all the loss components:
    1. manifold constraint: dual variable or loss weighting
    2. transport: how to estimate the transport field, and loss weighting
    3. critic: just the weight of the loss
    """
    constraint_config: ManifoldConstraintConfig
    """Constraint for manifold invariants"""
    transport_config: TransportLossConfig
    """Specifies how the transport field is estimated"""
    critic_loss_weight: float
    """Weight of the overall loss"""
    update_chart_every_n_critic_steps: int
    """Potentially update the critic many steps before updating the chart"""

In [ ]:
class ChartTransportModelConfig(BaseConfig):
    """
    The critic must be time-conditioned!
    Defaults to AdamW with standard weight-decay.
    """
    encoder: ModelConfig
    decoder: ModelConfig
    critic: ModelConfig

    chart_lr: float
    """Encoder-decoder LR"""
    critic_lr: float

In [ ]:
class ChartTransportConfig(BaseConfig):
    data_config: BaseDataConfig
    prior_config: BasePriorConfig
    loss_config: ChartTransportLossConfig
    model_config: ChartTransportModelConfig

In [ ]:
class TrainingConfig(BaseConfig):
    """
    Re-usable across methods, specifies method-agnostic training specifics
    """
    train_batch_size: int
    eval_batch_size: int